# Baseline ringkasan: kalimat awal
        **text › keywords_summary** · notebook `lead_baseline`

        **Kapan dipakai:** Pembanding wajib: pada berita, n kalimat pertama sering sudah kuat.

        **Jebakan umum:** Hanya cocok untuk teks berstruktur piramida terbalik (berita).

> Evaluasi di sini **kualitatif** (dibaca manusia). Metrik otomatis (ROUGE) butuh ringkasan referensi.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/text/classification_en/news4_train.csv"
TEST_PATH  = ROOT / "data_latihan/text/classification_en/news4_test.csv"
TEXT_COL   = "text"
LABEL_COL  = "label"            # hanya untuk EVALUASI; algoritma tidak melihatnya
LANG       = "en"
USE_STOPWORDS = True
SEED = 42
MODEL_KEY  = "lead_baseline"
OUT = out_dir("text", "keywords_summary", MODEL_KEY)
set_seed(SEED)
from superdb.text import ID_STOPWORDS
STOP = (ID_STOPWORDS if LANG == "id" else "english") if USE_STOPWORDS else None
DOC_IDX = [0, 5, 11]      # dokumen yang dicoba
TOP_K, N_SENT = 8, 3

## 2. Baca dokumen

In [ ]:
from superdb.text_extra import split_sentences, textrank_keywords, textrank_summary
df = pd.read_csv(TRAIN_PATH); docs = df[TEXT_COL].astype(str).tolist()
for i in DOC_IDX:
    print(f"[{i}] ({df[LABEL_COL][i]}) {len(docs[i].split())} kata, {len(split_sentences(docs[i]))} kalimat: {docs[i][:150]}...")

## 3. Jalankan metode

In [ ]:
result = {i: split_sentences(docs[i])[:N_SENT] for i in DOC_IDX}
for i, s in result.items():
    print(f"[{i}] ringkasan lead-{N_SENT} ({len(' '.join(s).split())}/{len(docs[i].split())} kata):"); [print("   -", x) for x in s]

## 4. Simpan

In [ ]:
save_json({str(k): v for k, v in result.items()}, OUT / "metrics.json"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Bandingkan metode berdampingan pada dokumen yang sama; **lead baseline wajib dikalahkan** sebelum mengklaim ringkasan lebih baik.
- Ada ringkasan referensi di data lomba? Hitung ROUGE (paket `rouge-score`; pasang lewat `tools/pipsafe.py`).
- Ringkasan abstraktif (menulis ulang) butuh model seq2seq (folder heavy).